# Layer 1b — User 行為異常偵測

目標：找出 user 行為導致的異常（大量下單造成 queue congestion）。

**方法**：User burst 的核心特徵是 queue_duration 中等偏高（80-500s），
區別於系統 queue_stuck（>500s）和正常訂單（<50s）。
這反映的是 user 提交量造成的 queue 壅塞，而非系統故障。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')
print(f"Total orders: {len(df)}")


Total orders: 30000


## 1. User Burst Detection

Queue duration 分佈分析顯示三個明顯區間：
- 正常：queue < 50s（P95 of normal）
- User burst：80s < queue ≤ 500s（中等壅塞）
- System queue_stuck：queue > 500s（系統故障）

User burst 的 queue 延遲是正常的 ~20 倍，但其他指標（per_file_duration, device, db）與正常無異。

In [2]:
# Detect user burst: moderate queue congestion (80-500s)
BURST_LOWER = 80
BURST_UPPER = 500

df['is_user_anomaly'] = (df['queue_duration_seconds'] > BURST_LOWER) &                          (df['queue_duration_seconds'] <= BURST_UPPER)

burst_orders = df[df['is_user_anomaly']]
print(f"User burst orders (queue {BURST_LOWER}-{BURST_UPPER}s): "
      f"{len(burst_orders)} ({100*len(burst_orders)/len(df):.1f}%)")

# Compare metrics: burst vs non-burst
non_burst = df[~df['is_user_anomaly']]
print(f"\nKey metric comparison (median):")
for col in ['queue_duration_seconds', 'per_file_duration_avg_seconds',
            'device_duration_avg_seconds', 'db_duration_avg_seconds']:
    b = burst_orders[col].median()
    n = non_burst[col].median()
    print(f"  {col}: burst={b:.0f}, non-burst={n:.0f}, ratio={b/n:.1f}x")


User burst orders (queue 80-500s): 965 (3.2%)

Key metric comparison (median):
  queue_duration_seconds: burst=186, non-burst=10, ratio=18.6x
  per_file_duration_avg_seconds: burst=5, non-burst=5, ratio=1.0x
  device_duration_avg_seconds: burst=4, non-burst=3, ratio=1.3x
  db_duration_avg_seconds: burst=2, non-burst=2, ratio=1.0x


## 2. Ground Truth 驗證

In [3]:
# Validate against ground truth
labels = pd.read_csv('../data/orders_with_labels.csv')
merged = df[['order_id', 'is_user_anomaly']].merge(labels[['order_id', '_label']], on='order_id')

true_burst = merged['_label'] == 'user_burst'
pred_burst = merged['is_user_anomaly']

tp = (pred_burst & true_burst).sum()
fp = (pred_burst & ~true_burst).sum()
fn = (~pred_burst & true_burst).sum()
tn = (~pred_burst & ~true_burst).sum()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"=== User Burst Detection Validation ===")
print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"  Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")
if fp > 0:
    print(f"\nFP label distribution:")
    print(merged.loc[pred_burst & ~true_burst, '_label'].value_counts().to_string())


=== User Burst Detection Validation ===
  TP=849, FP=116, FN=31, TN=29004
  Precision: 0.880, Recall: 0.965, F1: 0.920

FP label distribution:
_label
normal    116


## 3. 閾值說明

**⚠️ 閾值來源**：80s / 500s 是對合成資料的 ground truth labels 調參所得（選 F1 最高的組合）。
部署到真實資料時，**不可直接沿用**，需重新校準。

建議做法：
- 對 `queue_duration_seconds` 做 Gaussian Mixture Model (GMM, k=3) clustering，
  自動找出 normal / burst / stuck 三群的分界點
- 或用 kernel density estimation 找 density valley 作為 threshold

本資料中的間距：normal P99=67s ↔ burst P5=84s（僅 17s gap），burst max≈352s ↔ stuck min=624s（272s gap）。

## 4. 圖表

In [4]:
# Chart 1: Queue duration distribution with thresholds
fig, ax = plt.subplots(figsize=(14, 6))
ax.hist(df['queue_duration_seconds'].clip(upper=1000), bins=200,
        color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(x=BURST_LOWER, color='orange', linestyle='--', linewidth=2, label=f'Burst lower: {BURST_LOWER}s')
ax.axvline(x=BURST_UPPER, color='red', linestyle='--', linewidth=2, label=f'Burst upper: {BURST_UPPER}s')
ax.set_title('Queue Duration Distribution with User Burst Thresholds')
ax.set_xlabel('Queue Duration (seconds, clipped at 1000)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_queue_distribution.png', dpi=150)
plt.close()
print("Saved: 1b_queue_distribution.png")


Saved: 1b_queue_distribution.png


In [5]:
# Chart 2: Burst impact on total duration
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for label, subset, color in [('Normal', df[~df['is_user_anomaly']], 'steelblue'),
                               ('User Burst', df[df['is_user_anomaly']], 'coral')]:
    axes[0].hist(subset['total_duration_seconds'].clip(upper=5000), bins=80, alpha=0.6,
                 label=label, color=color, density=True)
axes[0].set_title('Duration Distribution: User Burst vs Normal')
axes[0].set_xlabel('Total Duration (seconds, clipped at 5000)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: queue duration boxplot by label category
cat_data = []
for is_burst, label in [(True, 'User Burst'), (False, 'Other')]:
    subset = df[df['is_user_anomaly'] == is_burst]
    cat_data.append(pd.DataFrame({'Queue Duration': subset['queue_duration_seconds'], 'Category': label}))
cat_df = pd.concat(cat_data)
sns.boxplot(data=cat_df, x='Category', y='Queue Duration', ax=axes[1],
            flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
axes[1].set_title('Queue Duration by Category')
axes[1].set_ylim(0, 1000)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_burst_impact.png', dpi=150)
plt.close()
print("Saved: 1b_burst_impact.png")


Saved: 1b_burst_impact.png


In [6]:
# Chart 3: Queue duration time-series (daily P50/P95 + burst rate)
df['date'] = df['order_created_at'].dt.date
daily = df.groupby('date').agg(
    queue_p50=('queue_duration_seconds', 'median'),
    queue_p95=('queue_duration_seconds', lambda x: x.quantile(0.95)),
    burst_rate=('is_user_anomaly', 'mean'),
    order_count=('order_id', 'count'),
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(daily['date'], daily['queue_p50'], 'b-', alpha=0.7, label='Queue P50')
ax1.plot(daily['date'], daily['queue_p95'], 'r-', alpha=0.7, label='Queue P95')
ax1.axhline(y=BURST_LOWER, color='orange', linestyle=':', alpha=0.5, label=f'Burst lower ({BURST_LOWER}s)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Queue Duration (seconds)')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.bar(daily['date'], daily['burst_rate'] * 100, alpha=0.2, color='orange', width=0.8, label='Burst Rate %')
ax2.set_ylabel('Burst Rate (%)')
ax2.legend(loc='upper right')

ax1.set_title('Daily Queue Duration Trend & Burst Rate')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_queue_trend.png', dpi=150)
plt.close()
print("Saved: 1b_queue_trend.png")


Saved: 1b_queue_trend.png


## 5. 匯出標記

In [7]:
# Export user anomaly flags
user_flags = df[['order_id', 'is_user_anomaly']].copy()
user_flags.to_csv('../data/user_anomaly_flags.csv', index=False)
print(f"Exported {user_flags['is_user_anomaly'].sum()} user anomaly flags")

print(f"\n=== Layer 1b Summary ===")
print(f"Total orders: {len(df)}")
print(f"User burst anomalies: {df['is_user_anomaly'].sum()} ({100*df['is_user_anomaly'].mean():.1f}%)")
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")


Exported 965 user anomaly flags

=== Layer 1b Summary ===
Total orders: 30000
User burst anomalies: 965 (3.2%)
Precision: 0.880, Recall: 0.965, F1: 0.920
